# 04 - Ratings per film and per year for the 142 nominees

Inputs: `data/raw/ml-32m/ratings.csv` and `data/processed/best_picture_nominees_1996_2017.csv` (built in notebook 03).
Output: `data/processed/film_year_window_Y.csv`.

**Main analysis window Y**: from the ceremony year to ceremony year + 5 (6 calendar years). The ceremony year is the first complete calendar year after a film's release, because every nominee was released the year before the ceremony.
**Robustness window X**: ceremony year + 1 to + 5 (excludes the ceremony year).
All years are computed in UTC.

In [1]:
import duckdb

con = duckdb.connect()

## A. A worked example: Titanic (movieId 1721) and platform activity

In [2]:
# Step 7: Titanic (movieId 1721), number of ratings and average rating per year (UTC)
con.sql("SELECT year(to_timestamp(timestamp) AT TIME ZONE 'UTC') AS rating_year, COUNT(*) AS nb_ratings, ROUND(AVG(rating), 2) AS avg_rating FROM '../data/raw/ml-32m/ratings.csv' WHERE movieId = 1721 GROUP BY rating_year ORDER BY rating_year").show(max_rows=40)

┌─────────────┬────────────┬────────────┐
│ rating_year │ nb_ratings │ avg_rating │
│    int64    │   int64    │   double   │
├─────────────┼────────────┼────────────┤
│        1997 │         39 │       4.31 │
│        1998 │       1542 │       4.11 │
│        1999 │       3525 │       3.32 │
│        2000 │       2929 │       3.58 │
│        2001 │       1826 │       3.51 │
│        2002 │       1378 │       3.28 │
│        2003 │       1189 │       3.27 │
│        2004 │       1986 │       3.15 │
│        2005 │       2805 │       3.06 │
│        2006 │       2211 │        3.2 │
│        2007 │       2316 │       3.13 │
│        2008 │       2556 │       3.22 │
│        2009 │       1669 │       3.28 │
│        2010 │       1629 │       3.31 │
│        2011 │       1287 │       3.41 │
│        2012 │       1006 │       3.49 │
│        2013 │        665 │       3.51 │
│        2014 │        528 │        3.4 │
│        2015 │       1878 │       3.44 │
│        2016 │       2881 │      

In [3]:
# Step 8: date and time (UTC) of the very first rating of Titanic (movieId 1721)
con.sql("SELECT to_timestamp(MIN(timestamp)) AT TIME ZONE 'UTC' AS first_rating_utc FROM '../data/raw/ml-32m/ratings.csv' WHERE movieId = 1721")

┌─────────────────────┐
│  first_rating_utc   │
│      timestamp      │
├─────────────────────┤
│ 1997-12-19 19:57:55 │
└─────────────────────┘

In [4]:
# Step 9: total number of ratings per year in the whole dataset (all movies), UTC
con.sql("SELECT year(to_timestamp(timestamp) AT TIME ZONE 'UTC') AS rating_year, COUNT(*) AS nb_ratings FROM '../data/raw/ml-32m/ratings.csv' GROUP BY rating_year ORDER BY rating_year").show(max_rows=40)

┌─────────────┬────────────┐
│ rating_year │ nb_ratings │
│    int64    │   int64    │
├─────────────┼────────────┤
│        1995 │          4 │
│        1996 │    1571368 │
│        1997 │     685388 │
│        1998 │     301691 │
│        1999 │    1174629 │
│        2000 │    1912322 │
│        2001 │    1160098 │
│        2002 │     849762 │
│        2003 │    1011000 │
│        2004 │    1139068 │
│        2005 │    1752573 │
│        2006 │    1141704 │
│        2007 │    1023813 │
│        2008 │    1117071 │
│        2009 │     890696 │
│        2010 │     860073 │
│        2011 │     729301 │
│        2012 │     695493 │
│        2013 │     563301 │
│        2014 │     518537 │
│        2015 │    1743866 │
│        2016 │    1918739 │
│        2017 │    1827953 │
│        2018 │    1391057 │
│        2019 │    1385467 │
│        2020 │    1688159 │
│        2021 │    1239912 │
│        2022 │     913105 │
│        2023 │     794054 │
├─────────────┴────────────┤
│ 29 rows     

## B. The table film x year for all nominees

In [5]:
# Step 11: for each of the 142 nominees and each rating year (UTC): number of ratings and average rating
con.sql("SELECT n.ceremony_year, n.title, n.is_winner, n.movieId, year(to_timestamp(r.timestamp) AT TIME ZONE 'UTC') AS rating_year, COUNT(*) AS nb_ratings, ROUND(AVG(r.rating), 2) AS avg_rating FROM '../data/processed/best_picture_nominees_1996_2017.csv' n JOIN '../data/raw/ml-32m/ratings.csv' r ON n.movieId = r.movieId GROUP BY n.ceremony_year, n.title, n.is_winner, n.movieId, rating_year ORDER BY n.ceremony_year, n.title, rating_year").show(max_rows=30)


┌───────────────┬───────────────────────┬───────────┬─────────┬─────────────┬────────────┬────────────┐
│ ceremony_year │         title         │ is_winner │ movieId │ rating_year │ nb_ratings │ avg_rating │
│     int64     │        varchar        │  boolean  │  int64  │    int64    │   int64    │   double   │
├───────────────┼───────────────────────┼───────────┼─────────┼─────────────┼────────────┼────────────┤
│          1996 │ Apollo 13             │ false     │     150 │        1996 │      19102 │        4.0 │
│          1996 │ Apollo 13             │ false     │     150 │        1997 │       1624 │       3.98 │
│          1996 │ Apollo 13             │ false     │     150 │        1998 │        990 │       4.04 │
│          1996 │ Apollo 13             │ false     │     150 │        1999 │       2024 │        4.0 │
│          1996 │ Apollo 13             │ false     │     150 │        2000 │       2397 │       4.02 │
│          1996 │ Apollo 13             │ false     │     150 │ 

In [6]:
# Step 12: create a reusable view with the number of ratings and average rating per nominee and per rating year (UTC)
con.sql("CREATE VIEW film_year AS SELECT n.ceremony_year, n.title, n.is_winner, n.movieId, year(to_timestamp(r.timestamp) AT TIME ZONE 'UTC') AS rating_year, COUNT(*) AS nb_ratings, ROUND(AVG(r.rating), 2) AS avg_rating FROM '../data/processed/best_picture_nominees_1996_2017.csv' n JOIN '../data/raw/ml-32m/ratings.csv' r ON n.movieId = r.movieId GROUP BY n.ceremony_year, n.title, n.is_winner, n.movieId, rating_year")

## C. How many film-year rows have few ratings?

Nothing is removed here, we only count. Step 16 uses window X (ceremony year + 1 to + 5); window Y gives the same counts of low-count rows (0, 2, 6, 28 below 10, 30, 50, 100 ratings).

In [7]:
# Step 15: how many film-year rows have fewer than N ratings, for several thresholds N (nothing is removed)
con.sql("SELECT COUNT(*) AS total_rows, SUM(CASE WHEN nb_ratings < 10 THEN 1 ELSE 0 END) AS below_10, SUM(CASE WHEN nb_ratings < 30 THEN 1 ELSE 0 END) AS below_30, SUM(CASE WHEN nb_ratings < 50 THEN 1 ELSE 0 END) AS below_50, SUM(CASE WHEN nb_ratings < 100 THEN 1 ELSE 0 END) AS below_100 FROM film_year")

┌────────────┬──────────┬──────────┬──────────┬───────────┐
│ total_rows │ below_10 │ below_30 │ below_50 │ below_100 │
│   int64    │  int128  │  int128  │  int128  │  int128   │
├────────────┼──────────┼──────────┼──────────┼───────────┤
│       2403 │       14 │       41 │      108 │       243 │
└────────────┴──────────┴──────────┴──────────┴───────────┘

In [8]:
# Step 16: same count, but only for the main analysis window (ceremony year + 1 to + 5)
con.sql("SELECT COUNT(*) AS total_rows, SUM(CASE WHEN nb_ratings < 10 THEN 1 ELSE 0 END) AS below_10, SUM(CASE WHEN nb_ratings < 30 THEN 1 ELSE 0 END) AS below_30, SUM(CASE WHEN nb_ratings < 50 THEN 1 ELSE 0 END) AS below_50, SUM(CASE WHEN nb_ratings < 100 THEN 1 ELSE 0 END) AS below_100 FROM film_year WHERE rating_year BETWEEN ceremony_year + 1 AND ceremony_year + 5")

┌────────────┬──────────┬──────────┬──────────┬───────────┐
│ total_rows │ below_10 │ below_30 │ below_50 │ below_100 │
│   int64    │  int128  │  int128  │  int128  │  int128   │
├────────────┼──────────┼──────────┼──────────┼───────────┤
│        710 │        0 │        2 │        6 │        28 │
└────────────┴──────────┴──────────┴──────────┴───────────┘

In [9]:
# Step 17: standard deviation of all ratings given to the 142 nominees
con.sql("SELECT ROUND(STDDEV(r.rating), 2) AS std_rating FROM '../data/processed/best_picture_nominees_1996_2017.csv' n JOIN '../data/raw/ml-32m/ratings.csv' r ON n.movieId = r.movieId")

┌────────────┐
│ std_rating │
│   double   │
├────────────┤
│       0.94 │
└────────────┘

## D. The main analysis window Y

In [10]:
# Step 19: view with only the main analysis window Y (ceremony year to ceremony year + 5), plus years since the ceremony
con.sql("CREATE VIEW analysis_window AS SELECT *, rating_year - ceremony_year AS years_since_ceremony FROM film_year WHERE rating_year BETWEEN ceremony_year AND ceremony_year + 5")

In [11]:
# check: the window has 6 years per film, 142 films each year since the ceremony
con.sql("SELECT years_since_ceremony, COUNT(*) AS n_rows FROM analysis_window GROUP BY years_since_ceremony ORDER BY years_since_ceremony")

┌──────────────────────┬────────┐
│ years_since_ceremony │ n_rows │
│        int64         │ int64  │
├──────────────────────┼────────┤
│                    0 │    142 │
│                    1 │    142 │
│                    2 │    142 │
│                    3 │    142 │
│                    4 │    142 │
│                    5 │    142 │
└──────────────────────┴────────┘

In [12]:
# Save the main window table (window Y) so that the next notebooks can load it directly
con.sql("COPY (SELECT ceremony_year, title, is_winner, movieId, rating_year, years_since_ceremony, nb_ratings, avg_rating FROM analysis_window ORDER BY ceremony_year, title, rating_year) TO '../data/processed/film_year_window_Y.csv' (HEADER, DELIMITER ',')")

In [13]:
# Check the saved file: 852 rows (142 films x 6 years), 142 movies, years_since_ceremony from 0 to 5
con.sql("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT movieId) AS n_films, MIN(years_since_ceremony) AS min_age, MAX(years_since_ceremony) AS max_age FROM '../data/processed/film_year_window_Y.csv'")

┌────────┬─────────┬─────────┬─────────┐
│ n_rows │ n_films │ min_age │ max_age │
│ int64  │  int64  │  int64  │  int64  │
├────────┼─────────┼─────────┼─────────┤
│    852 │     142 │       0 │       5 │
└────────┴─────────┴─────────┴─────────┘

## What we learned
- 2,403 film-year rows in total; 852 in window Y (142 films x 6 years, no gap).
- In window Y: 0 rows below 10 ratings, 2 below 30, 6 below 50, 28 below 100.
- The standard deviation of the ratings is 0.94, so the error of an average is about 0.94 / sqrt(n): 0.13 for 50 ratings, 0.09 for 100.
- Decisions: threshold 50 ratings (100 as robustness check), window Y as main window, window X as robustness check.